# Prueba clasificación de instituciones con modelos locales

In [1]:
import json
import re
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, '../src')
from bibliometrics.classification import system_prompt, inst_groups
from bibliometrics.config import MISSING

COUNTRY = "Honduras"
GROUPS = list(inst_groups(COUNTRY))
RESULTS_DIR = Path("local_model_results")
RESULTS_DIR.mkdir(exist_ok=True)

if sys.platform == "win32": # Keep Windows awake during long runs (released when the kernel stops)
    import ctypes
    ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001) # ES_CONTINUOUS | ES_SYSTEM_REQUIRED

In [2]:
inst_df = pd.read_excel("bibliometrics_honduras_db.xlsx", sheet_name="institutions", index_col=0, dtype=str)

In [3]:
inst_df.head()

,display_name,country_code,type,root_display_name,inst_group
institution_id,,,,,
I169009895,National Autonomous University of Honduras,HN,education,National Autonomous University of Honduras,Honduras
I4405272557,Hospital Escuela Universitario,HN,healthcare,Hospital Escuela Universitario,Honduras
I170897317,Duke University,US,education,Duke University,USA & Canada
I114027177,University of North Carolina at Chapel Hill,US,education,University of North Carolina System,USA & Canada
I1339895191,RWI – Leibniz Institute for Economic Research,DE,facility,Leibniz Association,Europe


In [4]:
inst_df.inst_group.value_counts()

inst_group
LAC                           278
USA & Canada                  228
Europe                        194
Other institutions             89
Multilateral organizations     33
Honduras                       16
Name: count, dtype: int64

## Objetivo y referencia

El pipeline clasifica cada institución única en 7 grupos con `claude-haiku-5-5`, lo que exige una `ANTHROPIC_API_KEY`. Aquí se prueba si un **modelo local** (sin API key y sin instalar programas, solo `pip`) o una **regla sin LLM** logra un desempeño equivalente.

- **Referencia**: las 838 etiquetas de Haiku de la corrida de Honduras (`inst_df.inst_group`). No son una verdad revisada por personas, así que los desacuerdos se revisan a mano al final.
- **Runtime local**: `llama-cpp-python` con modelos GGUF cuantizados (Q4_K_M) descargados de Hugging Face. Se instala con una wheel precompilada, sin compilador, Visual Studio ni token de HF:

```
pip install llama-cpp-python huggingface_hub --only-binary llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
```

- **Criterio de "desempeño equivalente"**: acuerdo ≥ 95 % con Haiku, recall de *Multilateral organizations* ≥ 90 %, 0 % sin clasificar y un tiempo razonable en un portátil sin GPU.

In [5]:
MULTI = "Multilateral organizations"

def per_class(pred, ref):
    ''' Precision, recall, F1 and support for each group '''
    rows = {}
    for g in GROUPS:
        tp, n_pred, n_ref = ((pred == g) & (ref == g)).sum(), (pred == g).sum(), (ref == g).sum()
        prec, rec = (tp / n_pred if n_pred else 0.0), (tp / n_ref if n_ref else float('nan'))
        rows[g] = {'precision': prec, 'recall': rec, 'f1': 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
                   'support': int(n_ref)}
    return pd.DataFrame(rows).T

def evaluate(pred, ref, seconds=None):
    ''' Agreement metrics of pred against ref (aligned Series) '''
    pred = pred.reindex(ref.index).fillna(MISSING)
    po = (pred == ref).mean()
    pe = sum(((pred == g).mean() * (ref == g).mean()) for g in GROUPS)
    pc = per_class(pred, ref)
    return {'agreement': po, 'kappa': (po - pe) / (1 - pe), 'macro_f1': pc.loc[pc.support > 0, 'f1'].mean(),
            'multi_precision': pc.loc[MULTI, 'precision'], 'multi_recall': pc.loc[MULTI, 'recall'],
            'unclassified': ((pred == MISSING) & (ref != MISSING)).mean(),
            'sec_per_inst': seconds / len(ref) if seconds else 0.0, 'n': len(ref)}

def confusion(pred, ref):
    return pd.crosstab(ref.rename('Haiku'), pred.reindex(ref.index).fillna(MISSING).rename('Predicción'))

evaluate(inst_df.inst_group, inst_df.inst_group) # Control: must be 1.0 agreement

{'agreement': np.float64(1.0),
 'kappa': np.float64(1.0),
 'macro_f1': np.float64(1.0),
 'multi_precision': np.float64(1.0),
 'multi_recall': np.float64(1.0),
 'unclassified': np.float64(0.0),
 'sec_per_inst': 0.0,
 'n': 838}

## Opción 0: reglas sin LLM

Región según `country_code` + lista de palabras clave de organismos multilaterales buscadas en `display_name` y `root_display_name`. No necesita dependencias ni modelo, y mide cuánto aporta realmente un LLM. Su límite es que la lista de multilaterales hay que mantenerla a mano.

In [6]:
COUNTRY_CODES = {"Honduras": "HN"}
LAC = set("AG AI AR AW BB BL BO BQ BR BS BZ CL CO CR CU CW DM DO EC FK GD GF GP GT GY HN HT JM KN KY LC MF MQ MS MX NI PA PE PR PY "
          "SR SV SX TC TT UY VC VE VG VI".split())
EUROPE = set("AD AL AT BA BE BG BY CH CY CZ DE DK EE ES FI FO FR GB GE GG GI GR HR HU IE IM IS IT JE LI LT LU LV MC MD ME MK MT "
             "NL NO PL PT RO RS SE SI SK SM UA VA XK".split()) # RU is left as "Other", like most Haiku labels
USA_CANADA = {"US", "CA"}

# Matched as case-insensitive substrings (keywords) or case-sensitive whole words (acronyms) in name and root.
# Caveat: written after seeing the Honduras multilaterals (INCAP, CGIAR, PANAFTOSA...), so its score here is optimistic.
MULTI_KEYWORDS = [
    "world bank", "inter-american development bank", "interamerican development bank", "banco interamericano de desarrollo",
    "international monetary fund", "fondo monetario internacional", "international finance corporation",
    "united nations", "naciones unidas", "nations unies", "children's fund", "world food programme",
    "food and agriculture organization", "world health organization", "organización mundial de la salud",
    "pan american health", "organización panamericana", "centro panamericano", "international labour", "international labor",
    "world trade organization", "organisation for economic co-operation", "organisation de coopération et de développement",
    "economic commission for latin america", "comisión económica para américa latina", "cgiar",
    "inter-american institute for cooperation on agriculture", "institute of nutrition of central america",
    "central american bank for economic integration", "banco centroamericano de integración",
    "sistema de la integración centroamericana", "organization of american states", "development bank of latin america",
    "international atomic energy", "international fund for agricultural development", "world meteorological organization",
    "international organization for migration", "international criminal court", "asian development bank",
    "african development bank", "world organisation for animal health",
]
MULTI_ACRONYMS = ["IDB", "BID", "IMF", "FMI", "OECD", "OCDE", "UNICEF", "UNDP", "PNUD", "UNESCO", "UNFPA", "UNHCR", "UNEP", "FAO",
                  "ILO", "OIT", "PAHO", "OPS", "CEPAL", "ECLAC", "IICA", "INCAP", "BCIE", "CABEI", "SICA", "IFAD", "IAEA", "WFP"]
MULTI_RE = re.compile("|".join([re.escape(k) for k in MULTI_KEYWORDS]), re.IGNORECASE)
ACRONYM_RE = re.compile(r"\b(" + "|".join(MULTI_ACRONYMS) + r")\b")

def classify_rules(df, country=COUNTRY, use_keywords=True):
    ''' Institution group with deterministic rules (use_keywords=False: region only) '''
    def one(row):
        text = f"{row.display_name} | {row.root_display_name}"
        if use_keywords and (MULTI_RE.search(text) or ACRONYM_RE.search(text)):
            return MULTI
        cc = row.country_code
        if not isinstance(cc, str) or cc == MISSING:
            return MISSING
        if cc == COUNTRY_CODES[country]:
            return country
        if cc in USA_CANADA:
            return "USA & Canada"
        if cc in LAC:
            return "LAC"
        if cc in EUROPE:
            return "Europe"
        return "Other institutions"
    return df.apply(one, axis=1)

t0 = time.time()
pred_rules = classify_rules(inst_df)
results = {'0. Reglas': evaluate(pred_rules, inst_df.inst_group, time.time() - t0),
           '0b. Solo región (sin palabras clave)': evaluate(classify_rules(inst_df, use_keywords=False), inst_df.inst_group)}
pd.DataFrame(results).T.round(3)

,agreement,kappa,macro_f1,multi_precision,multi_recall,unclassified,sec_per_inst,n
0. Reglas,0.996,0.995,0.997,1.0,1.0,0.0,0.0,838.0
0b. Solo región (sin palabras clave),0.957,0.942,0.814,0.0,0.0,0.0,0.0,838.0


In [7]:
# Disagreements of the rules: the cases that need knowledge (root abroad, Haiku inconsistency with RU)
inst_df[pred_rules != inst_df.inst_group].assign(rules=pred_rules)

,display_name,country_code,type,root_display_name,inst_group,rules
institution_id,,,,,,
I2802934987,United States Naval Medical Research Unit SOUTH,PE,healthcare,Government of the United States of America,USA & Canada,LAC
I4210126689,Research Centre for Medical Genetics,RU,healthcare,Russian Academy of Sciences,Europe,Other institutions
I4210102376,Centers for Disease Control and Prevention,UG,government,Centers for Disease Control and Prevention,USA & Canada,Other institutions


## Opciones 1–N: modelos locales con `llama-cpp-python`

- Los modelos GGUF Q4_K_M se descargan solos de Hugging Face la primera vez, a `%USERPROFILE%\.cache\huggingface`. No requieren token.
- **Una institución por llamada**: una gramática GBNF obliga a que la respuesta sea exactamente una de las 7 categorías. El prompt del sistema se reutiliza en la caché KV entre llamadas, así que cada llamada solo procesa ~50 tokens nuevos y genera 2–4.
- El formato por lotes del pipeline (lista JSON con `key`) se descartó: los modelos pequeños repetían o perdían keys.
- El prompt se arma con la **plantilla de chat del propio modelo y `enable_thinking=False`**. Sin esto, Qwen3 espera escribir `<think>`; como la gramática lo prohíbe, respondía al azar (0 % de acierto).
- Responder solo la etiqueta en vez de `{"inst_group": ...}` bajó la latencia de ~2,0–2,7 s a ~1,4–1,7 s por institución. El número de hilos (4, 6 u 8) no cambia la velocidad.

**Prueba previa de velocidad** (10 instituciones, salida JSON, i5-1235U sin GPU):

| Modelo | s/inst. | Decisión |
|---|---|---|
| Gemma 3 1B | 1,6 | Descartado: muy impreciso (3/10). Qwen3.5 0.8B además entra en bucle |
| Llama 3.2 3B, Gemma 3 4B, Qwen3 4B, Gemma 4 E2B, Phi-4-mini 3.8B, Granite 4.2 3B | 2,0–4,8 | Se evalúan |
| Gemma 4 E4B | 11,5 | Descartado: lento |
| Qwen3.5 4B | 18,4 | Descartado: lento. Su arquitectura híbrida no reutiliza la caché del prompt (838 inst. ≈ 4 h) |

**Prueba de prompts** (51 instituciones, 21 multilaterales; aciertos):

| Prompt | Llama 3.2 3B | Gemma 3 4B | Qwen3 4B |
|---|---|---|---|
| Corto (categorías del pipeline) | 39 | 32 | 37 |
| **Corto + ejemplos** (elegido) | 45 | 41 | **46** |
| Por pasos (multilateral → país) | 28 | 31 | 42 |
| Por pasos + ejemplos | 43 | 43 | 41 |
| Prompt del pipeline | 46 | 39 | 38 |

Los ejemplos del prompt (`EXAMPLES`) son instituciones que no están en los datos de Honduras.

In [8]:
import datetime
import os
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
from jinja2.sandbox import ImmutableSandboxedEnvironment
from llama_cpp import Llama, LlamaGrammar

# Development sample (~200): all multilaterals and Honduras, the rules' hard cases, and the rest stratified by group
hard = inst_df.index[pred_rules != inst_df.inst_group]
must = inst_df[(inst_df.inst_group.isin([MULTI, COUNTRY])) | inst_df.index.isin(hard)]
rest = inst_df.drop(must.index)
dev_df = pd.concat([must, rest.groupby('inst_group').sample(frac=(200 - len(must)) / len(rest), random_state=0)])
assert dev_df.inst_group.notna().all()
print(len(dev_df)); print(dev_df.inst_group.value_counts())

def choice_grammar(choices):
    ''' GBNF grammar that only allows one of the choices '''
    return LlamaGrammar.from_string("root ::= " + " | ".join(json.dumps(c, ensure_ascii=False) for c in choices), verbose=False)

def _raise(msg):
    raise ValueError(msg)

def chat_renderer(llm):
    ''' Prompt builder with the model's own chat template and thinking disabled (Qwen3 otherwise expects <think>,
    which the grammar forbids, and answers at random). BOS is added by the tokenizer '''
    template = ImmutableSandboxedEnvironment(trim_blocks=True, lstrip_blocks=True).from_string(llm.metadata['tokenizer.chat_template'])
    return lambda system, user: template.render(
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}], add_generation_prompt=True,
        enable_thinking=False, bos_token="", eos_token="", raise_exception=_raise,
        strftime_now=lambda fmt: datetime.datetime.now().strftime(fmt))

def inst_context(row):
    return json.dumps({field: None if row[col] == MISSING else row[col]
                       for field, col in {'name': 'display_name', 'country_code': 'country_code',
                                          'type': 'type', 'root': 'root_display_name'}.items()}, ensure_ascii=False)

def classify_llm(llm, df, system, choices=GROUPS):
    ''' One grammar-constrained call per institution (the shared system prefix stays in the KV cache).
    Returns (Series of answers, seconds) '''
    render, grammar = chat_renderer(llm), choice_grammar(choices)
    preds, t0 = {}, time.time()
    for inst_id, row in df.iterrows():
        out = llm.create_completion(render(system, inst_context(row)), grammar=grammar, temperature=0, max_tokens=16)
        answer = out["choices"][0]["text"].strip()
        preds[inst_id] = answer if answer in choices else MISSING
    return pd.Series(preds, name="pred"), time.time() - t0

def run_model(name, repo, filename, df, **kwargs):
    ''' Classify df with a GGUF model (cached in RESULTS_DIR). Returns (Series, seconds) '''
    path = RESULTS_DIR / f"{re.sub(r'[^A-Za-z0-9.]+', '_', name)}_{len(df)}.csv"
    if path.exists():
        cached = pd.read_csv(path, index_col=0)
        return cached['pred'], float(cached['seconds'].iloc[0])
    llm = Llama.from_pretrained(repo, filename, n_ctx=2048, verbose=False)
    pred, seconds = classify_llm(llm, df, **kwargs)
    del llm
    pred.to_frame().assign(seconds=seconds).to_csv(path)
    return pred, seconds

200
inst_group
LAC                           52
USA & Canada                  45
Europe                        37
Multilateral organizations    33
Other institutions            17
Honduras                      16
Name: count, dtype: int64


In [9]:
MODELS = { # name: (Hugging Face repo, GGUF file)
    "Llama 3.2 3B": ("bartowski/Llama-3.2-3B-Instruct-GGUF", "Llama-3.2-3B-Instruct-Q4_K_M.gguf"),
    "Gemma 3 4B": ("unsloth/gemma-3-4b-it-GGUF", "gemma-3-4b-it-Q4_K_M.gguf"),
    "Qwen3 4B": ("Qwen/Qwen3-4B-GGUF", "Qwen3-4B-Q4_K_M.gguf"),
    "Gemma 4 E2B": ("unsloth/gemma-4-E2B-it-GGUF", "gemma-4-E2B-it-Q4_K_M.gguf"),
    "Phi-4-mini 3.8B": ("unsloth/Phi-4-mini-instruct-GGUF", "Phi-4-mini-instruct-Q4_K_M.gguf"),
    "Granite 4.2 3B": ("ibm-granite/granite-4.2-3b-GGUF", "granite-4.2-3b-Q4_K_M.gguf"),
}

# Few-shot examples that are NOT in the Honduras data
EXAMPLES = [
    ({"name": "Asian Development Bank", "country_code": "PH", "type": "other", "root": "Asian Development Bank"}, MULTI),
    ({"name": "International Rice Research Institute", "country_code": "PH", "type": "nonprofit", "root": "CGIAR"}, MULTI),
    ({"name": "University of Toronto", "country_code": "CA", "type": "education", "root": "University of Toronto"}, "USA & Canada"),
    ({"name": "Ministry of Health", "country_code": "PH", "type": "government", "root": "Government of the Philippines"}, "Other institutions"),
    ({"name": "Universidad de Chile", "country_code": "CL", "type": "education", "root": "Universidad de Chile"}, "LAC"),
    ({"name": "Wellcome Trust", "country_code": "GB", "type": "funder", "root": "Wellcome Trust"}, "Europe"),
    ({"name": "Secretaría de Salud", "country_code": COUNTRY_CODES[COUNTRY], "type": "government", "root": f"Government of {COUNTRY}"}, COUNTRY),
]

def classify_prompt(country):
    ''' "fewshot" prompt: category definitions of the pipeline + examples. Best average of the prompt tests (see markdown) '''
    categories = system_prompt(country).split("Categorías:")[1].split("Reglas:")[0]
    examples = "\n".join(f"{json.dumps(e, ensure_ascii=False)} -> {g}" for e, g in EXAMPLES)
    return (f"Clasifica el origen de una institución que participa en estudios académicos sobre {country}.\n"
            f"Categorías:{categories}Responde solo con el nombre exacto de la categoría.\n\nEjemplos:\n{examples}")

CLASSIFY_PROMPT = classify_prompt(COUNTRY)
dev_ref = dev_df.inst_group
dev_results, dev_preds = {'0. Reglas': evaluate(pred_rules[dev_df.index], dev_ref)}, {}
for name, (repo, filename) in MODELS.items():
    dev_preds[name], seconds = run_model(name, repo, filename, dev_df, system=CLASSIFY_PROMPT)
    dev_results[name] = evaluate(dev_preds[name], dev_ref, seconds)
    print(f"{name}: {dev_results[name]['agreement']:.3f} ({dev_results[name]['sec_per_inst']:.2f} s/inst)")

pd.DataFrame(dev_results).T.sort_values('agreement', ascending=False).round(3)

Llama 3.2 3B: 0.900 (1.33 s/inst)
Gemma 3 4B: 0.785 (1.61 s/inst)
Qwen3 4B: 0.940 (2.02 s/inst)


C:\Users\andre\Documents_local\BID_2026\Reporte_bibliometrico\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


Gemma 4 E2B: 0.870 (1.03 s/inst)
Phi-4-mini 3.8B: 0.800 (1.47 s/inst)
Granite 4.2 3B: 0.745 (1.63 s/inst)


,agreement,kappa,macro_f1,multi_precision,multi_recall,unclassified,sec_per_inst,n
0. Reglas,0.985,0.981,0.983,1.000,1.000,0.000,0.000,200.0
Qwen3 4B,0.940,0.925,0.925,0.935,0.879,0.005,2.019,200.0
Llama 3.2 3B,0.900,0.876,0.899,0.889,0.727,0.005,1.333,200.0
Gemma 4 E2B,0.870,0.842,0.865,0.939,0.939,0.000,1.026,200.0
Phi-4-mini 3.8B,0.800,0.749,0.701,0.838,0.939,0.000,1.469,200.0
Gemma 3 4B,0.785,0.733,0.685,0.900,0.818,0.015,1.606,200.0
Granite 4.2 3B,0.745,0.685,0.698,0.517,0.939,0.005,1.631,200.0


## Opción híbrida: reglas para la región + LLM solo para detectar multilaterales

La región sale casi siempre de `country_code`; lo único que exige conocimiento es reconocer organismos multilaterales, cuyo `country_code` es el de su sede. El híbrido:

1. Usa las reglas de región **sin** la lista de palabras clave, para medir si el LLM la puede reemplazar y generalizar a otros países.
2. Le pregunta al LLM `{"multilateral": true/false}` solo para los tipos de OpenAlex donde aparecen multilaterales (`government`, `nonprofit`, `other`, `facility`, `funder`). Universidades, hospitales y empresas se excluyen, así que son ~210 llamadas en vez de 838.
3. Variante "+ palabras clave": une la detección del LLM con la lista de palabras clave.

In [10]:
MULTI_CANDIDATE_TYPES = {"government", "nonprofit", "other", "facility", "funder"}
MULTI_EXAMPLES = [ # Not in the Honduras data
    ({"name": "Asian Development Bank", "country_code": "PH", "type": "other", "root": "Asian Development Bank"}, "sí"),
    ({"name": "International Rice Research Institute", "country_code": "PH", "type": "nonprofit", "root": "CGIAR"}, "sí"),
    ({"name": "UNESCO Office in Quito", "country_code": "EC", "type": "government", "root": "UNESCO"}, "sí"),
    ({"name": "Wellcome Trust", "country_code": "GB", "type": "funder", "root": "Wellcome Trust"}, "no"),
    ({"name": "Ministry of Health", "country_code": "PH", "type": "government", "root": "Government of the Philippines"}, "no"),
    ({"name": "United States Agency for International Development", "country_code": "US", "type": "government",
      "root": "Government of the United States of America"}, "no"),
    ({"name": "Bill & Melinda Gates Foundation", "country_code": "US", "type": "funder", "root": "Bill & Melinda Gates Foundation"}, "no"),
]
MULTI_PROMPT = ("Decide si una institución es un organismo multilateral o intergubernamental: BID/IDB, Banco Mundial, OCDE, FMI, "
                "Naciones Unidas y sus agencias (PNUD, UNICEF, FAO, OIT, OMS/OPS, CEPAL...), centros del CGIAR, BCIE, SICA, OEA y similares, "
                "incluidas sus oficinas nacionales o regionales. Las universidades, ministerios, agencias de cooperación de un solo país, "
                "ONG y fundaciones privadas NO son multilaterales.\n"
                "Recibes un JSON con su \"name\", \"country_code\" (ISO-2, a menudo el de su sede), \"type\" (tipo de OpenAlex) y \"root\" "
                "(organización padre raíz). Responde solo \"sí\" o \"no\".\n\nEjemplos:\n"
                + "\n".join(f"{json.dumps(e, ensure_ascii=False)} -> {a}" for e, a in MULTI_EXAMPLES))

def classify_hybrid(name, df, with_keywords=False):
    ''' Region rules + LLM multilateral detection on candidate types. Returns (Series, seconds) '''
    region = classify_rules(df, use_keywords=False)
    candidates = df[df.type.isin(MULTI_CANDIDATE_TYPES)]
    is_multi, seconds = run_model(f"multi {name}", *MODELS[name], candidates, system=MULTI_PROMPT, choices=["sí", "no"])
    multi = df.index.isin(is_multi[is_multi == "sí"].index)
    if with_keywords:
        multi |= (classify_rules(df) == MULTI).to_numpy()
    return region.where(~multi, MULTI), seconds

In [11]:
for name in MODELS:
    for kw in (False, True):
        pred, seconds = classify_hybrid(name, dev_df, with_keywords=kw)
        label = f"Híbrido {name}" + (" + palabras clave" if kw else "")
        dev_preds[label], dev_results[label] = pred, evaluate(pred, dev_ref, seconds)

dev_summary = pd.DataFrame(dev_results).T.sort_values('agreement', ascending=False)
dev_summary.round(3)

C:\Users\andre\Documents_local\BID_2026\Reporte_bibliometrico\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


,agreement,kappa,macro_f1,multi_precision,multi_recall,unclassified,sec_per_inst,n
0. Reglas,0.985,0.981,0.983,1.000,1.000,0.000,0.000,200.0
Híbrido Granite 4.2 3B + palabras clave,0.985,0.981,0.983,1.000,1.000,0.000,0.475,200.0
Híbrido Llama 3.2 3B + palabras clave,0.980,0.975,0.978,0.971,1.000,0.000,0.353,200.0
Híbrido Gemma 4 E2B + palabras clave,0.980,0.975,0.976,0.971,1.000,0.000,0.306,200.0
Híbrido Granite 4.2 3B,0.965,0.957,0.963,1.000,0.879,0.000,0.475,200.0
Híbrido Gemma 4 E2B,0.965,0.957,0.960,0.968,0.909,0.000,0.306,200.0
Híbrido Qwen3 4B,0.960,0.950,0.958,0.846,1.000,0.000,0.561,200.0
Híbrido Qwen3 4B + palabras clave,0.960,0.950,0.958,0.846,1.000,0.000,0.561,200.0
Qwen3 4B,0.940,0.925,0.925,0.935,0.879,0.005,2.019,200.0
Híbrido Llama 3.2 3B,0.940,0.926,0.938,0.962,0.758,0.000,0.353,200.0


## Corrida completa (838 instituciones) con las mejores opciones

In [12]:
FULL_MODELS = ["Qwen3 4B"] # Best model classifying everything in the development run
FULL_HYBRID = ["Granite 4.2 3B", "Gemma 4 E2B", "Qwen3 4B"] # Best hybrids without keywords

ref = inst_df.inst_group
preds = {'0. Reglas': pred_rules}
for name in FULL_MODELS:
    preds[name], seconds = run_model(name, *MODELS[name], inst_df, system=CLASSIFY_PROMPT)
    results[name] = evaluate(preds[name], ref, seconds)
for name in FULL_HYBRID:
    for kw in (False, True):
        label = f"Híbrido {name}" + (" + palabras clave" if kw else "")
        preds[label], seconds = classify_hybrid(name, inst_df, with_keywords=kw)
        results[label] = evaluate(preds[label], ref, seconds)

summary = pd.DataFrame(results).T.sort_values('agreement', ascending=False)
summary['minutes_total'] = summary.sec_per_inst * summary.n / 60
summary.round(3)

C:\Users\andre\Documents_local\BID_2026\Reporte_bibliometrico\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


C:\Users\andre\Documents_local\BID_2026\Reporte_bibliometrico\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


C:\Users\andre\Documents_local\BID_2026\Reporte_bibliometrico\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


C:\Users\andre\Documents_local\BID_2026\Reporte_bibliometrico\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


,agreement,kappa,macro_f1,multi_precision,multi_recall,unclassified,sec_per_inst,n,minutes_total
0. Reglas,0.996,0.995,0.997,1.000,1.000,0.000,0.000,838.0,0.001
Híbrido Granite 4.2 3B + palabras clave,0.990,0.987,0.982,0.868,1.000,0.000,0.402,838.0,5.620
Híbrido Gemma 4 E2B + palabras clave,0.990,0.987,0.981,0.868,1.000,0.000,0.271,838.0,3.784
Híbrido Gemma 4 E2B,0.987,0.982,0.972,0.857,0.909,0.000,0.271,838.0,3.784
Híbrido Granite 4.2 3B,0.986,0.981,0.969,0.853,0.879,0.000,0.402,838.0,5.620
Híbrido Qwen3 4B,0.958,0.945,0.923,0.500,1.000,0.000,0.481,838.0,6.712
Híbrido Qwen3 4B + palabras clave,0.958,0.945,0.923,0.500,1.000,0.000,0.481,838.0,6.712
0b. Solo región (sin palabras clave),0.957,0.942,0.814,0.000,0.000,0.000,0.000,838.0,0.000
Qwen3 4B,0.934,0.912,0.882,0.690,0.879,0.004,2.030,838.0,28.346


In [13]:
# Disagreements with Haiku for manual review: fill "correcto" with "haiku", "local" or "dudoso"
review = inst_df.assign(**{name: pred.reindex(inst_df.index) for name, pred in preds.items()})
review = review[(review[list(preds)].ne(review.inst_group, axis=0)).any(axis=1)].assign(correcto="")
review.to_csv(RESULTS_DIR / "desacuerdos_revision.csv", encoding="utf-8-sig")
print(len(review), "instituciones con algún desacuerdo")
review

80 instituciones con algún desacuerdo


,display_name,country_code,type,root_display_name,inst_group,0. Reglas,Qwen3 4B,Híbrido Granite 4.2 3B,Híbrido Granite 4.2 3B + palabras clave,Híbrido Gemma 4 E2B,Híbrido Gemma 4 E2B + palabras clave,Híbrido Qwen3 4B,Híbrido Qwen3 4B + palabras clave,correcto
institution_id,,,,,,,,,,,,,,
I1288051870,Organisation de Coopération et de Développemen...,FR,government,Organisation de Coopération et de Développemen...,Multilateral organizations,Multilateral organizations,Europe,Multilateral organizations,Multilateral organizations,Multilateral organizations,Multilateral organizations,Multilateral organizations,Multilateral organizations,
I1288059949,International Finance Corporation,KZ,government,International Finance Corporation,Multilateral organizations,Multilateral organizations,Multilateral organizations,Other institutions,Multilateral organizations,Other institutions,Multilateral organizations,Multilateral organizations,Multilateral organizations,
I4387153627,International Criminal Court,NL,government,International Criminal Court,Multilateral organizations,Multilateral organizations,Europe,Europe,Multilateral organizations,Multilateral organizations,Multilateral organizations,Multilateral organizations,Multilateral organizations,
I1313323035,Russian Academy of Sciences,RU,government,Russian Academy of Sciences,Other institutions,Other institutions,Europe,Other institutions,Other institutions,Other institutions,Other institutions,Other institutions,Other institutions,
I19880235,Lomonosov Moscow State University,RU,education,Lomonosov Moscow State University,Other institutions,Other institutions,Europe,Other institutions,Other institutions,Other institutions,Other institutions,Other institutions,Other institutions,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
I27765905,North-West University,ZA,education,North-West University,Other institutions,Other institutions,Europe,Other institutions,Other institutions,Other institutions,Other institutions,Other institutions,Other institutions,
I99043593,Macquarie University,AU,education,Macquarie University,Other institutions,Other institutions,USA & Canada,Other institutions,Other institutions,Other institutions,Other institutions,Other institutions,Other institutions,
I98734588,Social Sciences and Humanities Research Council,CA,government,Social Sciences and Humanities Research Council,USA & Canada,USA & Canada,USA & Canada,USA & Canada,USA & Canada,USA & Canada,USA & Canada,Multilateral organizations,Multilateral organizations,
